# 16 — Bảng kết quả khảo sát và proof

    Chỉ Run All khi tất cả suite đã chọn hoàn tất. Khóa model/config/ngưỡng trước test; không tuning trên test ở đây. Base month7 đã được quan sát ở các notebook cũ nên là development test, không gọi là tập xác nhận chưa từng mở. Bảng tách từng backbone, giữ kết quả thực tế kể cả phương pháp không tốt hơn baseline.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = Path(
    os.environ.get("BAF_CACHE_ROOT") or ROOT / ".cache" / "baf-research"
).expanduser().resolve()
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Project: D:\UIT\DS317\BAF_project
Module: baf-research-20261028-v1


In [2]:
# Điền đường dẫn in ở notebook 11 nếu đã có nhiều campaign.
# Ví dụ: Path(r"D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_...")
CAMPAIGN_DIR = None
CAMPAIGN = gru.research_resolve_campaign(ROOT, CAMPAIGN_DIR)
PLAN = gru.research_read(CAMPAIGN / "campaign.json")

print("Campaign:", CAMPAIGN)
print("Mode:", PLAN["mode"])
print("Suites:", PLAN["suites"])
print("Seeds:", PLAN["config"]["seeds"])


Campaign: D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e
Mode: full
Suites: ['losses', 'meta', 'lsr', 'xgboost']
Seeds: [42, 317, 2026]


In [3]:
for suite in PLAN["suites"]:
    marker = CAMPAIGN / "suites" / suite / "FROZEN.json"
    if not marker.is_file():
        raise RuntimeError(f"Chưa xong suite {suite}; hoàn tất notebook tương ứng trước16.")
    print("FROZEN:", suite)


FROZEN: losses
FROZEN: meta
FROZEN: lsr
FROZEN: xgboost


## 1. Đánh giá và bootstrap trong notebook thực thi riêng

    Lưu score test từng model, ngưỡng validation đã khóa, FPR thực tế/fairness và paired-row bootstrap ΔAP. Cùng hàng bootstrap cho mọi cặp seed. CI có điều kiện trên model đã fit; SD seed không thay thế CI test. Rerun chỉ kiểm và đọc kết quả đã hoàn tất.

In [4]:
EVALUATION = gru.research_launch_evaluation(CAMPAIGN)
print("Evaluation:", EVALUATION)


BEGIN_STAGE 1
ENTIRE_RESEARCH_TABLE_SEALED_BEFORE_TEST 60
RESEARCH_STUDY losses mode= full device= cuda
TEST_METRICS {'suite': 'losses', 'method': 'balanced', 'seed': 42, 'comparison_group': 'shared_mlp', 'threshold': 0.8309551477432251, 'fit_seconds': 119.98292770000262, 'mode': 'full', 'ap': 0.18898316539589993, 'roc_auc': 0.8809588945885957, 'recall_at_fpr5_curve': 0.5280112044817927, 'recall_frozen_threshold': 0.5630252100840336, 'fpr_frozen_threshold': 0.059382696640989364, 'precision_frozen_threshold': 0.12426584234930448, 'alert_rate_frozen_threshold': 0.06680916535010274, 'recall_at_0p5': 0.8396358543417367, 'precision_at_0p5': 0.04854644100736902, 'f1_weighted_at_0p5': 0.847087036924743, 'accuracy_at_0p5': 0.7549848724223743, 'tn': 89749, 'fp': 5666, 'fn': 624, 'tp': 804, 'younger_fpr': 0.04761561214919559, 'younger_tpr': 0.49306930693069306, 'younger_n': 84176, 'older_fpr': 0.1392766756469916, 'older_tpr': 0.7320574162679426, 'older_n': 12667, 'fpr_gap_older_minus_younger': 0

## 2. Bảng theo phương pháp và paper

    Không ghép AP của notebook07/08/09 vào đây vì split/preprocessing/backbone/meta holdout khác. Cột adaptation phải đi cùng kết quả. Đọc PDF để bổ sung tập dữ liệu/kết quả gốc và khoảng trống; bảng số liệu tự nó chưa chứng minh tính mới.

In [5]:
import pandas as pd

table = pd.read_csv(EVALUATION / "research_table.csv")
paired = pd.read_csv(EVALUATION / "paired_bootstrap.csv")
display(table)
display(paired)

paper_table = gru.research_paper_table(CAMPAIGN)
display(paper_table)
print("CSV:", EVALUATION / "paper_research_table.csv")


,comparison_group,method,ap_mean,ap_std,recall_frozen_threshold_mean,recall_frozen_threshold_std,fpr_frozen_threshold_mean,fpr_frozen_threshold_std,precision_frozen_threshold_mean,precision_frozen_threshold_std,older_fpr_mean,older_fpr_std,younger_fpr_mean,younger_fpr_std,fpr_gap_older_minus_younger_mean,fpr_gap_older_minus_younger_std,fpr_ratio_min_max_mean,fpr_ratio_min_max_std,fit_seconds_mean,fit_seconds_std
0,lsr_latent_mlp,lsr_nn_balanced,0.206689,0.000795,0.577264,0.007488,0.057776,0.001136,0.130093,0.001085,0.128800,0.002641,0.047315,0.000962,0.081485,0.001915,0.367371,0.004348,175.449676,50.619809
1,lsr_latent_mlp,lsr_nn_none,0.210313,0.002021,0.580999,0.022630,0.057168,0.004889,0.132317,0.006093,0.127657,0.006584,0.046786,0.004825,0.080871,0.004185,0.366169,0.025908,146.779588,23.079415
2,lsr_latent_mlp,lsr_nn_weighted,0.209832,0.003332,0.581699,0.018559,0.056766,0.005892,0.133481,0.008241,0.125833,0.011291,0.046594,0.005122,0.079240,0.006356,0.369898,0.010734,174.135012,54.874439
3,lsr_raw_lightgbm,lsr_lgbm_balanced,0.215188,0.001472,0.548786,0.003454,0.044563,0.001543,0.155699,0.003784,0.107165,0.001990,0.035343,0.001477,0.071822,0.000513,0.329703,0.007730,22.436869,1.901229
4,lsr_raw_lightgbm,lsr_lgbm_none,0.215599,0.001694,0.553221,0.003501,0.043438,0.000635,0.160105,0.001772,0.107220,0.001473,0.034044,0.000649,0.073175,0.001469,0.317547,0.006577,21.066894,1.011320
5,lsr_raw_lightgbm,lsr_lgbm_weighted,0.216167,0.002181,0.549486,0.005617,0.043704,0.000552,0.158376,0.001677,0.107356,0.000432,0.034329,0.000659,0.073027,0.000931,0.319779,0.006778,21.523908,0.604368
6,normalized_head_mlp,ldam,0.184456,0.001831,0.526144,0.010856,0.050541,0.003659,0.135060,0.005975,0.119901,0.003271,0.040325,0.003769,0.079576,0.001826,0.335992,0.023577,172.462259,0.627395
7,normalized_head_mlp,ldam_drw,0.184617,0.001655,0.534314,0.002801,0.052259,0.002078,0.132803,0.003951,0.120690,0.004694,0.042181,0.002004,0.078509,0.004240,0.349669,0.016130,176.188346,5.128626
8,normalized_head_mlp,norm_ce,0.191185,0.003940,0.525444,0.030575,0.049500,0.006695,0.137906,0.009867,0.116173,0.006684,0.039680,0.006785,0.076493,0.003047,0.340276,0.042983,191.759548,37.902981
9,shared_mlp,balanced,0.190752,0.002329,0.557656,0.007014,0.056567,0.002942,0.128712,0.004495,0.132283,0.011207,0.045415,0.003790,0.086868,0.013367,0.345843,0.051024,124.844264,4.284111


,comparison_group,reference,method,delta_ap_mean,delta_ap_ci_low,delta_ap_bootstrap_median,delta_ap_ci_high,bootstrap_repeats,bootstrap_kind
0,lsr_latent_mlp,lsr_nn_none,lsr_nn_balanced,-0.003624,-0.007933,-0.003572,0.000859,1000,paired IID rows; same resample across seeds; c...
1,lsr_latent_mlp,lsr_nn_none,lsr_nn_weighted,-0.000481,-0.002986,-0.000420,0.002312,1000,paired IID rows; same resample across seeds; c...
2,lsr_raw_lightgbm,lsr_lgbm_none,lsr_lgbm_balanced,-0.000411,-0.005601,-0.000501,0.004985,1000,paired IID rows; same resample across seeds; c...
3,lsr_raw_lightgbm,lsr_lgbm_none,lsr_lgbm_weighted,0.000569,-0.001056,0.000611,0.002229,1000,paired IID rows; same resample across seeds; c...
4,normalized_head_mlp,norm_ce,ldam,-0.006729,-0.010421,-0.006623,-0.003312,1000,paired IID rows; same resample across seeds; c...
5,normalized_head_mlp,norm_ce,ldam_drw,-0.006568,-0.010075,-0.006457,-0.003287,1000,paired IID rows; same resample across seeds; c...
6,shared_mlp,ce,balanced,-0.004473,-0.008657,-0.004460,-0.000358,1000,paired IID rows; same resample across seeds; c...
7,shared_mlp,ce,cb_ce,-0.005066,-0.010215,-0.004885,-0.000028,1000,paired IID rows; same resample across seeds; c...
8,shared_mlp,ce,cb_focal,-0.006615,-0.011931,-0.006494,-0.001028,1000,paired IID rows; same resample across seeds; c...
9,shared_mlp,ce,drw,-0.000846,-0.002089,-0.000813,0.000402,1000,paired IID rows; same resample across seeds; c...


,comparison_group,method,ap_mean,ap_std,recall_frozen_threshold_mean,recall_frozen_threshold_std,fpr_frozen_threshold_mean,fpr_frozen_threshold_std,precision_frozen_threshold_mean,precision_frozen_threshold_std,...,seeds,evidence_path,interpretation,reference,delta_ap_mean,delta_ap_ci_low,delta_ap_bootstrap_median,delta_ap_ci_high,bootstrap_repeats,bootstrap_kind
0,lsr_latent_mlp,lsr_nn_balanced,0.206689,0.000795,0.577264,0.007488,0.057776,0.001136,0.130093,0.001085,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,lsr_nn_none,-0.003624,-0.007933,-0.003572,0.000859,1000.0,paired IID rows; same resample across seeds; c...
1,lsr_latent_mlp,lsr_nn_none,0.210313,0.002021,0.580999,0.022630,0.057168,0.004889,0.132317,0.006093,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lsr_latent_mlp,lsr_nn_weighted,0.209832,0.003332,0.581699,0.018559,0.056766,0.005892,0.133481,0.008241,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,lsr_nn_none,-0.000481,-0.002986,-0.000420,0.002312,1000.0,paired IID rows; same resample across seeds; c...
3,lsr_raw_lightgbm,lsr_lgbm_balanced,0.215188,0.001472,0.548786,0.003454,0.044563,0.001543,0.155699,0.003784,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,lsr_lgbm_none,-0.000411,-0.005601,-0.000501,0.004985,1000.0,paired IID rows; same resample across seeds; c...
4,lsr_raw_lightgbm,lsr_lgbm_none,0.215599,0.001694,0.553221,0.003501,0.043438,0.000635,0.160105,0.001772,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,lsr_raw_lightgbm,lsr_lgbm_weighted,0.216167,0.002181,0.549486,0.005617,0.043704,0.000552,0.158376,0.001677,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,lsr_lgbm_none,0.000569,-0.001056,0.000611,0.002229,1000.0,paired IID rows; same resample across seeds; c...
6,normalized_head_mlp,ldam,0.184456,0.001831,0.526144,0.010856,0.050541,0.003659,0.135060,0.005975,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,norm_ce,-0.006729,-0.010421,-0.006623,-0.003312,1000.0,paired IID rows; same resample across seeds; c...
7,normalized_head_mlp,ldam_drw,0.184617,0.001655,0.534314,0.002801,0.052259,0.002078,0.132803,0.003951,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,norm_ce,-0.006568,-0.010075,-0.006457,-0.003287,1000.0,paired IID rows; same resample across seeds; c...
8,normalized_head_mlp,norm_ce,0.191185,0.003940,0.525444,0.030575,0.049500,0.006695,0.137906,0.009867,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,shared_mlp,balanced,0.190752,0.002329,0.557656,0.007014,0.056567,0.002942,0.128712,0.004495,...,"[42, 317, 2026]",D:\UIT\DS317\BAF_project\results\research_camp...,Observed BAF result; no claim of reproducing o...,ce,-0.004473,-0.008657,-0.004460,-0.000358,1000.0,paired IID rows; same resample across seeds; c...


CSV: D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e\evaluation\paper_research_table.csv


## 3. Xuất proof đánh giá

    Đây là proof của giai đoạn đánh giá. Giữ thêm full proof của12–15 để có toàn bộ checkpoint/training. Review ZIP có score/nhãn/code/output và chỉ xác minh các file được giữ. Chưa chạy CCRW-E hay cải tiến đề xuất trong gói khảo sát này.

In [6]:
PROOF = gru.research_export_proof(EVALUATION, ROOT)
for kind, path in PROOF.items():
    print(kind, "→", path)


RESEARCH_TEST_TABLE_VERIFIED 60
FULL_PROOF D:\UIT\DS317\BAF_project\results\proofs\evaluation_e0cc7f8f_full_proof.zip
PARTIAL_REVIEW D:\UIT\DS317\BAF_project\results\proofs\evaluation_e0cc7f8f_review.zip
full_proof → D:\UIT\DS317\BAF_project\results\proofs\evaluation_e0cc7f8f_full_proof.zip
review → D:\UIT\DS317\BAF_project\results\proofs\evaluation_e0cc7f8f_review.zip
